In [9]:
import pandas as pd
import sys
# Agregar la ruta raíz para que el notebook encuentre la carpeta src y el config.yaml
sys.path.append('../') 

from src.features.build_features import FeatureEngineer

# Instanciar el módulo leyendo el YAML
ingeniero = FeatureEngineer(config_path='../config.yaml')

# (Aquí cargarías un DataFrame 'df' de prueba)
# df_procesado = ingeniero.aplicar_pca(df)
# df_procesado.head()

In [10]:
import pandas as pd

# Cargamos uno de tus archivos (ajusta el nombre al mes que quieras probar)
df = pd.read_csv('../data/raw/datos_abiertos_2015_01.csv')

# Vemos las primeras líneas para confirmar que cargó bien
df.head()

,Viaje_Id,Usuario_Id,Genero,Año_de_nacimiento,Inicio_del_viaje,Fin_del_viaje,Origen_Id,Destino_Id
0,42034,17636,M,1989.0,2015-01-01 00:03:06,2015-01-01 00:05:52,19,19
1,42046,10701,M,1986.0,2015-01-01 06:00:24,2015-01-01 06:03:53,37,25
2,42047,13525,M,1983.0,2015-01-01 07:06:08,2015-01-01 07:06:59,55,55
3,42048,17625,M,1980.0,2015-01-01 07:25:14,2015-01-01 07:33:13,81,54
4,42049,272,M,1980.0,2015-01-01 07:36:56,2015-01-01 07:48:33,79,9


In [11]:
import sys
sys.path.append('../')

# Importar la clase que acabamos de crear
from src.models.train_model import MiBiciPipeline

# NOTA PARA TI: Asegúrate de tener cargado tu DataFrame en la variable 'df' antes de correr esto.
# 1. Usar el FeatureEngineer (Punto 2) para quitar las columnas inútiles
df_limpio = ingeniero.limpiar_columnas_inutiles(df)

# 2. Separar solo los datos numéricos para el modelo y rellenar nulos
X = df_limpio.drop(columns=ingeniero.col_id, errors='ignore').select_dtypes(include='number')
X = X.fillna(X.median())

# 3. Instanciar el Pipeline (Punto 3)
pipeline_ml = MiBiciPipeline(config_path='../config.yaml')

# 4. Entrenar, Evaluar y Guardar (Automatizado)
pipeline_ml.entrenar(X)
predicciones = pipeline_ml.evaluar(X)

# Guardar el modelo en la carpeta 'models/' de Cookiecutter
pipeline_ml.guardar_modelo(ruta_guardado='../models/pipeline_mibici.pkl')

AttributeError: 'MiBiciPipeline' object has no attribute 'entrenar'

In [12]:
import sys
sys.path.append('../')
from src.models.train_model import MiBiciPipeline

# Instanciar el pipeline actualizado
pipeline_ml = MiBiciPipeline(config_path='../config.yaml')

# 1. Ajuste de hiperparámetros (Tuning - Iterará sobre la malla del config.yaml)
print("--- FASE DE AJUSTE (TUNING) ---")
pipeline_ml.optimizar_hiperparametros(X)

# 2. Evaluación con métricas de rendimiento (Silhouette Score)
print("\n--- FASE DE EVALUACIÓN ---")
predicciones_optimizadas = pipeline_ml.evaluar_modelo(X)

# 3. Guardado del modelo optimizado
pipeline_ml.guardar_modelo(ruta_guardado='../models/pipeline_optimizado_mibici.pkl')

--- FASE DE AJUSTE (TUNING) ---
Iniciando ajuste de hiperparámetros...
Probando n_estimators=50, contamination=auto -> Silhouette Score: 0.4109
Probando n_estimators=50, contamination=0.05 -> Silhouette Score: 0.5884
Probando n_estimators=50, contamination=0.1 -> Silhouette Score: 0.5306
Probando n_estimators=100, contamination=auto -> Silhouette Score: 0.4109
Probando n_estimators=100, contamination=0.05 -> Silhouette Score: 0.5884
Probando n_estimators=100, contamination=0.1 -> Silhouette Score: 0.5392
Probando n_estimators=150, contamination=auto -> Silhouette Score: 0.4109
Probando n_estimators=150, contamination=0.05 -> Silhouette Score: 0.5884
Probando n_estimators=150, contamination=0.1 -> Silhouette Score: 0.5386

Mejores hiperparámetros encontrados: {'n_estimators': 50, 'contamination': 0.05} con Score: 0.5884

--- FASE DE EVALUACIÓN ---

--- REPORTE DE EVALUACIÓN ---
Silhouette Score (Rendimiento de separación): 0.5865
Viajes normales detectados: 30600
Anomalías detectadas: 1

In [ ]:
import mlflow
import mlflow.sklearn
from sklearn.metrics import silhouette_score

# 1. Configurar MLflow con base de datos SQLite (Obligatorio para el Model Registry)
mlflow.set_tracking_uri("sqlite:///mlruns.db")
mlflow.set_experiment("Proyecto_MiBici_Deteccion_Anomalias")

print("Iniciando registro de experimento en MLflow...")

# 2. Iniciar el "Run" de seguimiento
with mlflow.start_run(run_name="Isolation_Forest_Optimizado") as run:
    
    # Parámetros ganadores de tu Punto 4
    mejores_params = {'n_estimators': 50, 'contamination': 0.05}
    
    # Entrenar el pipeline final usando la clase que ya tienes
    pipeline_final = pipeline_ml.crear_pipeline(**mejores_params)
    pipeline_final.fit(X)
    
    # Evaluación para guardar los resultados
    predicciones = pipeline_final.predict(X)
    score = silhouette_score(X, predicciones, sample_size=10000, random_state=42)
    anomalias = (predicciones == -1).sum()
    
    # 3. Registrar hiperparámetros y métricas (Tracking)
    mlflow.log_params(mejores_params)
    mlflow.log_metric("silhouette_score", score)
    mlflow.log_metric("anomalias_detectadas", anomalias)
    
    # 4. Guardar y registrar el modelo (Model Registry)
    mlflow.sklearn.log_model(
        sk_model=pipeline_final,
        artifact_path="modelo_isolation_forest",
        registered_model_name="MiBici_IsolationForest_Model",
        skops_trusted_types=["sklearn.tree._tree.Tree"]  # <-- Línea agregada para autorizar el modelo
    )
    
    print(f"Experimento guardado con éxito. Run ID: {run.info.run_id}")
    print("Modelo guardado en la versión 1 del MLflow Model Registry.")

2026/11/07 07:58:43 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/11/07 07:58:43 INFO mlflow.store.db.utils: Updating database tables
2026/11/07 07:59:35 INFO mlflow.tracking.fluent: Experiment with name 'Proyecto_MiBici_Deteccion_Anomalias' does not exist. Creating a new experiment.


Iniciando registro de experimento en MLflow...


2026/11/07 07:59:48 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


MlflowException: The saved sklearn model references untrusted types. If you are sure loading these types is safe, set the 'skops_trusted_types' parameter when calling 'log_model' or 'save_model' to the list of trusted types. Root error: Untrusted types found in the file: ['sklearn.tree._tree.Tree'].

- sklearn.tree._tree.Tree: sklearn.tree._tree.Tree (the shared node storage for DecisionTree*, RandomForest*, ExtraTrees*, and GradientBoosting* models) stores raw node indices (left_child, right_child, feature) that scikit-learn indexes into without bounds checking. A malicious file can set these to out-of-range values: skops loads the object successfully, but calling .predict() on it can then crash the process (segfault) or read out-of-bounds memory. If you created the file yourself or otherwise fully trust its source, you can load it with trusted=["sklearn.tree._tree.Tree"].

Only add the specific types you have reviewed and trust to the `trusted` argument; avoid passing everything repo